In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
# from pyspark.sql.window import Window

import os 
import sys

project_pth = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)
from utils.transformations import reusable

In [0]:
# Read stream from bronze
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimUser/schema") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@saspotifyy.dfs.core.windows.net/DimUser")

# Apply transformations
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

# Write stream to delta table
df_user.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimUser/checkpoint") \
    .trigger(once=True) \
    .toTable("spotify.silver.DimUser")

In [0]:
# display(
#     df_user,
#     checkpointLocation="abfss://silver@saspotifyy.dfs.core.windows.net/DimUser/debug_checkpoint"
# )
# # dbutils.fs.rm("abfss://silver@saspotifyy.dfs.core.windows.net/DimUser/debug_checkpoint", True)

In [0]:
# Read stream from bronze
df_artist = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimArtist/schema") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@saspotifyy.dfs.core.windows.net/DimArtist")

# Apply transformations
df_art_obj = reusable()

df_artist = df_art_obj.dropColumns(df_artist, ['_rescued_data'])
# df_artist = df_artist.dropDuplicates(['artist_id'])

# Write stream to delta table
df_artist.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimArtist/checkpoint") \
    .trigger(once=True) \
    .toTable("spotify.silver.DimArtist")

In [0]:
# display(
#     df_artist,
#     checkpointLocation="abfss://silver@saspotifyy.dfs.core.windows.net/DimArtist/debug_checkpoint"
# )
# # dbutils.fs.rm("abfss://silver@saspotifyy.dfs.core.windows.net/DimArtist/debug_checkpoint", True)

In [0]:
# Read stream from bronze
df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimTrack/schema") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@saspotifyy.dfs.core.windows.net/DimTrack")

# Apply transformations
df_track_obj = reusable()

df_track = df_track_obj.dropColumns(df_track, ['_rescued_data'])
df_track = df_track.dropDuplicates(['track_id'])



In [0]:
df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")\
                                            .when(col('duration_sec')<300,"medium")\
                                            .otherwise("high"))

df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))
# Write stream to delta table
df_track.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimTrack/checkpoint") \
    .trigger(once=True) \
    .toTable("spotify.silver.DimTrack")

In [0]:
# Read stream from bronze
df_date = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimDate/schema") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@saspotifyy.dfs.core.windows.net/DimDate")

# Apply transformations
df_date_obj = reusable()

df_date = df_date_obj.dropColumns(df_date, ['_rescued_data'])
# Write stream to delta table
df_date.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/DimDate/checkpoint") \
    .trigger(once=True) \
    .toTable("spotify.silver.DimDate")

In [0]:
# Read stream from bronze
df_fact = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/FactStream/schema") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@saspotifyy.dfs.core.windows.net/FactStream")

# Apply transformations
df_fact_obj = reusable()

df_fact = df_fact_obj.dropColumns(df_fact, ['_rescued_data'])
# Write stream to delta table
df_fact.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@saspotifyy.dfs.core.windows.net/FactStream/checkpoint") \
    .trigger(once=True) \
    .toTable("spotify.silver.FactStream")